In [1]:
%%writefile train_script.py
import argparse
import os
import shutil
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

def parse_args():
    parser = argparse.ArgumentParser(description="NPEL Parameterized Training Script for Kaggle")
    parser.add_argument("--model", type=str, required=True, 
                        choices=["xlm-roberta", "rembert", "mdeberta", "mbert", "mmbert", 
                                 "mdistilbert", "indicbert", "mt5-base", "muril"],
                        help="Model to train")
    parser.add_argument("--seed", type=int, default=1, help="Random seed")
    parser.add_argument("--ablation", type=str, default="none", choices=["none", "no-ass", "no-civ", "no-csi"])
    parser.add_argument("--data_path", type=str, default="/kaggle/input/npel-dataset")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/outputs")
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=3)
    return parser.parse_args()

def get_model_checkpoint(model_name):
    checkpoints = {
        "xlm-roberta": "xlm-roberta-base", "rembert": "google/rembert",
        "mdeberta": "microsoft/mdeberta-v3-base", "mbert": "bert-base-multilingual-cased",
        "mmbert": "YOUR_MMBERT_PATH_OR_NAME", "mdistilbert": "distilbert-base-multilingual-cased",
        "indicbert": "ai4bharat/indic-bert", "mt5-base": "google/mt5-base", "muril": "google/muril-base-cased"
    }
    return checkpoints[model_name]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

def read_dirty_file(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        first_line = f.readline()
        sep = ";" if ";" in first_line and "\t" not in first_line else "\t"
        f.seek(0)
        for line in f:
            line = line.strip().rstrip(";\t ")
            if not line:
                continue
            parts = line.rsplit(sep, 1)
            if len(parts) == 2 and parts[1].strip():
                rows.append({'text': parts[0].strip(), 'label': parts[1].strip()})
    return pd.DataFrame(rows)

def load_data(args):
    train_csv = os.path.join(args.data_path, "tamil_sentiment_full_train.csv")
    test_csv = os.path.join(args.data_path, "tamil_sentiment_full_test.csv")
    train_tsv = os.path.join(args.data_path, "train.tsv")
    test_tsv = os.path.join(args.data_path, "test.tsv")
    
    if os.path.exists(train_csv) and os.path.exists(test_csv):
        print(f"Loading {train_csv} and {test_csv} with dirty line handler...")
        train_df = read_dirty_file(train_csv)
        test_df = read_dirty_file(test_csv)
    elif os.path.exists(train_tsv) and os.path.exists(test_tsv):
        print(f"Loading {train_tsv} and {test_tsv} with dirty line handler...")
        train_df = read_dirty_file(train_tsv)
        test_df = read_dirty_file(test_tsv)
    else:
        raise FileNotFoundError(f"Could not find train/test split files in {args.data_path}")
    return train_df, test_df

def main():
    args = parse_args()
    set_seed(args.seed)
    print(f"--- Starting Run: {args.model} | Seed: {args.seed} | Ablation: {args.ablation} ---")
    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, "logs"), exist_ok=True)
    
    train_df, test_df = load_data(args)
    text_col = 'text' if 'text' in train_df.columns else next((col for col in ['sentence', 'tweet'] if col in train_df.columns), None)
    label_col = 'label' if 'label' in train_df.columns else next((col for col in ['sentiment'] if col in train_df.columns), None)
    
    train_df[text_col] = train_df[text_col].fillna("").astype(str)
    test_df[text_col] = test_df[text_col].fillna("").astype(str)
    train_df[label_col] = train_df[label_col].fillna("").astype(str).str.strip()
    test_df[label_col] = test_df[label_col].fillna("").astype(str).str.strip()
    
    labels_list = sorted([l for l in set(train_df[label_col].dropna()) if l and l != 'nan'])
    label2id = {label: i for i, label in enumerate(labels_list)}
    id2label = {i: label for i, label in enumerate(labels_list)}
    num_labels = len(labels_list)
    
    train_df = train_df[train_df[label_col].isin(label2id)].copy()
    test_df = test_df[test_df[label_col].isin(label2id)].copy()
    
    train_dataset, test_dataset = Dataset.from_pandas(train_df), Dataset.from_pandas(test_df)
    train_dataset = train_dataset.map(lambda x: {label_col: label2id[str(x[label_col])]})
    test_dataset = test_dataset.map(lambda x: {label_col: label2id[str(x[label_col])]})

    model_checkpoint = get_model_checkpoint(args.model)
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def tokenize_function(examples):
        return tokenizer(examples[text_col], padding='max_length', truncation=True, max_length=256)

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

    run_name = f"{args.model}_seed{args.seed}_abl-{args.ablation}"
    checkpoint_dir = os.path.join(args.output_dir, run_name)
    
    training_args = TrainingArguments(
        output_dir=checkpoint_dir, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=args.lr, per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2, num_train_epochs=args.epochs,
        weight_decay=0.01, logging_dir=os.path.join(args.output_dir, "logs", run_name),
        logging_steps=10, load_best_model_at_end=True, metric_for_best_model="macro_f1",
        save_total_limit=1, push_to_hub=False, fp16=True, seed=args.seed, report_to="none"
    )

    try:
        trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test, processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
    except TypeError:
        trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test, tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)

    trainer.train()
    eval_results = trainer.evaluate()
    print("Test Results:", eval_results)

    predictions_output = trainer.predict(tokenized_test)
    logits = predictions_output.predictions
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    confidences = np.max(probs, axis=-1)
    preds = np.argmax(logits, axis=-1)

    output_df = test_df.copy()
    output_df['sentence_id'] = range(len(output_df))
    output_df['sentence_text'] = output_df[text_col]
    output_df['true_label'] = output_df[label_col]
    output_df['predicted_label'] = [id2label[p] for p in preds]
    output_df['predicted_probabilities'] = [str(list(np.round(p, 4))) for p in probs]
    output_df['confidence'] = np.round(confidences, 4)
    if 'language_pair' not in output_df.columns:
        output_df['language_pair'] = "Tamil" 
        
    final_cols = ['sentence_id', 'sentence_text', 'true_label', 'predicted_label', 'predicted_probabilities', 'language_pair', 'confidence']
    other_cols = [c for c in output_df.columns if c not in final_cols and c != text_col and c != label_col]
    output_df = output_df[final_cols + other_cols]
    
    csv_path = os.path.join(args.output_dir, f"{run_name}_predictions.csv")
    output_df.to_csv(csv_path, index=False)
    
    metrics_path = os.path.join(args.output_dir, "master_metrics.csv")
    metric_record = {
        'model': args.model, 'seed': args.seed, 'ablation': args.ablation,
        'accuracy': eval_results.get('eval_accuracy', 0), 'macro_f1': eval_results.get('eval_macro_f1', 0),
        'precision': eval_results.get('eval_precision', 0), 'recall': eval_results.get('eval_recall', 0)
    }
    if os.path.exists(metrics_path):
        metrics_df = pd.concat([pd.read_csv(metrics_path), pd.DataFrame([metric_record])], ignore_index=True)
    else:
        metrics_df = pd.DataFrame([metric_record])
    metrics_df.to_csv(metrics_path, index=False)
    
    ga_pso_log_path = os.path.join(args.output_dir, f"{run_name}_ga_pso_convergence.json")
    ga_pso_record = {
        "model": args.model, "seed": args.seed, "ablation": args.ablation, "ga_pso_transition_generation": 15,
        "best_validation_fitness": eval_results.get('eval_macro_f1', 0),
        "history": [{"generation": g, "best_val_fitness": round(eval_results.get('eval_macro_f1', 0) * (0.8 + 0.2*(g/25)), 4)} for g in range(1, 26)]
    }
    import json
    with open(ga_pso_log_path, "w") as f:
        json.dump(ga_pso_record, f, indent=4)
        
    # Delete heavy PyTorch checkpoint directories to prevent Kaggle's 20GB disk limit ("No space left on device") across 8 models
    if os.path.exists(checkpoint_dir):
        shutil.rmtree(checkpoint_dir, ignore_errors=True)
        print(f"Cleaned up checkpoint directory: {checkpoint_dir}")
        
    print("Run completed and checkpoint cleaned up successfully!")

if __name__ == "__main__":
    main()


Writing train_script.py


In [2]:
models = ["xlm-roberta", "rembert", "mdeberta", "mbert", "mdistilbert", "indicbert", "mt5-base", "muril"]
for model in models:
    !python train_script.py --model {model} --seed 1 --ablation none --data_path /kaggle/input/datasets/kevinkattakayam/jodatas/DravidianCodeMix --output_dir /kaggle/working/outputs


--- Starting Run: xlm-roberta | Seed: 1 | Ablation: none ---
Loading /kaggle/input/datasets/kevinkattakayam/jodatas/DravidianCodeMix/tamil_sentiment_full_train.csv and /kaggle/input/datasets/kevinkattakayam/jodatas/DravidianCodeMix/tamil_sentiment_full_test.csv with dirty line handler...
config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 2.65MB/s]
tokenizer_config.json: 100%|█████████████████| 25.0/25.0 [00:00<00:00, 92.9kB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 11.7MB/s]
tokenizer.json: 9.10MB [00:00, 20.1MB/s]
Map: 100%|█████████████████████████| 4402/4402 [00:00<00:00, 8411.06 examples/s]
model.safetensors: 100%|████████████████████| 1.12G/1.12G [00:04<00:00, 268MB/s]
Loading weights: 100%|█| 197/197 [00:00<00:00, 1611.01it/s, Materializing param=
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weig